# Prompt Management with Vertex AI SDK

------------------------------------
Detailed Jupyter notebook tutorial on managing prompts using Vertex AI
Includes: creating, versioning, listing, modifying, inference, deletion, and tagging


## Setup


In [2]:
!pip install --upgrade google-cloud-aiplatform --quiet

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.30.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.41.0 which is incompatible.


In [1]:
import os
from google.cloud import aiplatform
from vertexai import Client, types
from google import genai
from google.genai import types as genai_types


In [2]:
PROJECT = "bdc-trainings"
LOCATION = "us-central1"
CLIENT = Client(project=PROJECT, location=LOCATION)
print(f"Vertex AI client created for project {PROJECT}, location {LOCATION}")


Vertex AI client created for project bdc-trainings, location us-central1


## Register the prompt to Vertex AI



In [3]:
## Define a local prompt

define_prompt = types.Prompt(
    prompt_data=types.PromptData(
        contents=[
            genai_types.Content(parts=[genai_types.Part(text="Hello, {name}! How are you doing today?")])
        ],
        variables=[{"name": genai_types.Part(text="Alice")}, {"name": genai_types.Part(text="Bob")}],
        model="models/gemini-2.0-flash",
    )
)
print("Local prompt defined.")



Local prompt defined.


In [4]:
registered_prompt = CLIENT.prompts.create(prompt=define_prompt)
print("Prompt registered with ID:", registered_prompt.prompt_id)


RefreshError: ('Failed to retrieve http://metadata.google.internal/computeMetadata/v1/instance/service-accounts/default/?recursive=true from the Google Compute Engine metadata service. Status: 404 Response:\nb\'<!DOCTYPE html>\\n<html lang=en>\\n  <meta charset=utf-8>\\n  <meta name=viewport content="initial-scale=1, minimum-scale=1, width=device-width">\\n  <title>Error 404 (Not Found)!!1</title>\\n  <style>\\n    *{margin:0;padding:0}html,code{font:15px/22px arial,sans-serif}html{background:#fff;color:#222;padding:15px}body{margin:7% auto 0;max-width:390px;min-height:180px;padding:30px 0 15px}* > body{background:url(//www.google.com/images/errors/robot.png) 100% 5px no-repeat;padding-right:205px}p{margin:11px 0 22px;overflow:hidden}ins{color:#777;text-decoration:none}a img{border:0}@media screen and (max-width:772px){body{background:none;margin-top:0;max-width:none;padding-right:0}}#logo{background:url(//www.google.com/images/branding/googlelogo/1x/googlelogo_color_150x54dp.png) no-r

## Modify and create a new version


In [ ]:
modified_prompt = types.Prompt(
    prompt_data=types.PromptData(
        contents=[
            genai_types.Content(parts=[genai_types.Part(text="Hi, {name}! Welcome back. How may I help you today?")])
        ],
        variables=[{"name": genai_types.Part(text="Charlie")}, {"name": genai_types.Part(text="Dana")}],
        model="gemini-2.0-flash",
    )
)
new_version = CLIENT.prompts.create_version(prompt=modified_prompt, prompt_id=registered_prompt.prompt_id)
print("New prompt version created:", new_version.version_id)


New prompt version created: 2


## Retrieve a prompt and its version, run inference

In [ ]:
retrieved_prompt = CLIENT.prompts.get(prompt_id=registered_prompt.prompt_id)
print("Retrieved prompt ID:", retrieved_prompt.prompt_id)

retrieved_version = CLIENT.prompts.get_version(
    prompt_id=registered_prompt.prompt_id, version_id=new_version.version_id
)
print("Retrieved prompt version:", retrieved_version.version_id)


Retrieved prompt ID: 6009056995435347968
Retrieved prompt version: 2


In [ ]:
# List prompts and versions

print("Listing prompts in project:")
for p in CLIENT.prompts.list():
    print(" •", p.prompt_id)

print(f"Listing versions for prompt {registered_prompt.prompt_id}:")
for v in CLIENT.prompts.list_versions(prompt_id=registered_prompt.prompt_id):
    print(" • Version ID:", v.version_id)


Listing prompts in project:
 • 6009056995435347968
 • 8475903691327537152
 • 7668549895237664768
Listing versions for prompt 6009056995435347968:
 • Version ID: 1
 • Version ID: 2


In [ ]:
#  Run inference with a loaded prompt
genai_client = genai.Client(vertexai=True, project=PROJECT, location=LOCATION)
contents = retrieved_version.prompt_data.contents
assembled_prompt = retrieved_version.assemble_contents()
response = genai_client.models.generate_content(
    model=retrieved_version.prompt_data.model, contents=assembled_prompt
)
print("Generated response:", response)


Generated response: sdk_http_response=HttpResponse(
  headers=<dict len=9>
) candidates=[Candidate(
  avg_logprobs=-0.5732574462890625,
  content=Content(
    parts=[
      Part(
        text="""It seems like you're giving me the same greeting twice, once as "Charlie" and once as "Dana." 

Since I'm a language model, I don't have a personal name or a returning status. To best help you, please tell me what you need assistance with!  For example, you could ask me a question, give me a writing prompt, or ask me to summarize something.
"""
      ),
    ],
    role='model'
  ),
  finish_reason=<FinishReason.STOP: 'STOP'>
)] create_time=datetime.datetime(2025, 11, 10, 5, 19, 31, 9063, tzinfo=TzInfo(0)) model_version='gemini-2.0-flash' prompt_feedback=None response_id='43URaedGg_CoyA_Vl-bwBQ' usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=86,
  candidates_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=86
 

In [ ]:
# Label or alias a prompt version
alias_prompt = CLIENT.prompts.create_version(
    prompt=types.Prompt(
        prompt_data=types.PromptData(
            contents=[
                genai_types.Content(parts=[genai_types.Part(text="Production: Hello {name}, how may I assist you today?")])
            ],
            variables=[{"name": genai_types.Part(text="Eve")}, {"name": genai_types.Part(text="Frank")}],
            model="models/text-generation-model-name",
        ),
    ),
    prompt_id=registered_prompt.prompt_id,
    config=types.CreatePromptConfig(
        display_name="my-prompt-prod",
        labels={"environment": "production"},
    ),
)
print("Alias version created:", alias_prompt.version_id, alias_prompt.labels)


## Delete a prompt version and entire prompt


In [ ]:
CLIENT.prompts.delete_version(
    prompt_id=registered_prompt.prompt_id, version_id=new_version.version_id
)
print(f"Deleted version {new_version.version_id}")

CLIENT.prompts.delete(prompt_id=registered_prompt.prompt_id)
print(f"Deleted prompt {registered_prompt.prompt_id}")


In [ ]:
# Restore a previous version
restored = CLIENT.prompts.restore_version(
    prompt_id=registered_prompt.prompt_id, version_id="your-version-id-to-restore"
)
print("Restored version ID:", restored.version_id)

